In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/df_long.csv', parse_dates=['date'])
df = df.sort_values(['id', 'date']).reset_index(drop=True)

print("Shape:", df.shape)
print("Date range:", df['date'].min(), "to", df['date'].max())
df.head(3)

Shape: (388200, 18)
Date range: 2011-01-29 00:00:00 to 2016-05-22 00:00:00


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wday,month,year,event_name_1,event_type_1,snap_CA,wm_yr_wk,sell_price,day_name
0,FOODS_1_218_CA_1_evaluation,FOODS_1_218,FOODS_1,FOODS,CA_1,CA,d_1,10,2011-01-29,1,1,2011,NaN,NaN,0,11101,1.0,Sun
1,FOODS_1_218_CA_1_evaluation,FOODS_1_218,FOODS_1,FOODS,CA_1,CA,d_2,3,2011-01-30,2,1,2011,NaN,NaN,0,11101,1.0,Mon
2,FOODS_1_218_CA_1_evaluation,FOODS_1_218,FOODS_1,FOODS,CA_1,CA,d_3,4,2011-01-31,3,1,2011,NaN,NaN,0,11101,1.0,Tue


In [2]:
# Lag features = what were sales X days ago?
# We use 7, 14, 28 because retail has weekly cycles
# shift(7) means "sales from 7 days ago for this same item"

for lag in [7, 14, 28]:
    df[f'lag_{lag}'] = df.groupby('id')['sales'].shift(lag)

print("Lag features created")
print(df[['id', 'date', 'sales', 'lag_7', 'lag_14', 'lag_28']].head(15))

Lag features created
                             id       date  sales  lag_7  lag_14  lag_28
0   FOODS_1_218_CA_1_evaluation 2011-01-29     10    NaN     NaN     NaN
1   FOODS_1_218_CA_1_evaluation 2011-01-30      3    NaN     NaN     NaN
2   FOODS_1_218_CA_1_evaluation 2011-01-31      4    NaN     NaN     NaN
3   FOODS_1_218_CA_1_evaluation 2011-02-01      0    NaN     NaN     NaN
4   FOODS_1_218_CA_1_evaluation 2011-02-02      0    NaN     NaN     NaN
5   FOODS_1_218_CA_1_evaluation 2011-02-03      0    NaN     NaN     NaN
6   FOODS_1_218_CA_1_evaluation 2011-02-04      1    NaN     NaN     NaN
7   FOODS_1_218_CA_1_evaluation 2011-02-05      4   10.0     NaN     NaN
8   FOODS_1_218_CA_1_evaluation 2011-02-06      2    3.0     NaN     NaN
9   FOODS_1_218_CA_1_evaluation 2011-02-07      0    4.0     NaN     NaN
10  FOODS_1_218_CA_1_evaluation 2011-02-08      0    0.0     NaN     NaN
11  FOODS_1_218_CA_1_evaluation 2011-02-09      4    0.0     NaN     NaN
12  FOODS_1_218_CA_1_evaluatio

In [3]:
# Rolling mean = average sales over last N days
# shift(1) before rolling ensures we don't leak today's sales into the feature

df['rolling_mean_7'] = (
    df.groupby('id')['sales']
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

df['rolling_mean_28'] = (
    df.groupby('id')['sales']
    .transform(lambda x: x.shift(1).rolling(28).mean())
)

df['rolling_std_7'] = (
    df.groupby('id')['sales']
    .transform(lambda x: x.shift(1).rolling(7).std())
)

print("Rolling features created")
df[['id', 'date', 'sales', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7']].head(35).tail(5)

Rolling features created


,id,date,sales,rolling_mean_7,rolling_mean_28,rolling_std_7
30,FOODS_1_218_CA_1_evaluation,2011-02-28,0,0.000000,0.821429,0.000000
31,FOODS_1_218_CA_1_evaluation,2011-03-01,26,0.000000,0.678571,0.000000
32,FOODS_1_218_CA_1_evaluation,2011-03-02,14,3.714286,1.607143,9.827076
33,FOODS_1_218_CA_1_evaluation,2011-03-03,24,5.714286,2.107143,10.355583
34,FOODS_1_218_CA_1_evaluation,2011-03-04,16,9.142857,2.964286,11.992061


In [4]:
# Extract time-based features from date
df['day_of_week'] = df['date'].dt.dayofweek      # 0=Monday, 6=Sunday
df['day_of_month'] = df['date'].dt.day
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
df['is_month_end'] = df['date'].dt.is_month_end.astype(int)

print("Calendar features created")
df[['date', 'day_of_week', 'is_weekend', 'month', 'week_of_year']].head(5)

Calendar features created


,date,day_of_week,is_weekend,month,week_of_year
0,2011-01-29,5,1,1,4
1,2011-01-30,6,1,1,4
2,2011-01-31,0,0,1,5
3,2011-02-01,1,0,2,5
4,2011-02-02,2,0,2,5


In [5]:
# Is there a major event today?
df['is_event'] = df['event_name_1'].notna().astype(int)

# Encode event type
df['is_sporting_event'] = (df['event_type_1'] == 'Sporting').astype(int)
df['is_national_event'] = (df['event_type_1'] == 'National').astype(int)
df['is_religious_event'] = (df['event_type_1'] == 'Religious').astype(int)
df['is_cultural_event'] = (df['event_type_1'] == 'Cultural').astype(int)

print("Event features created")
print("Total event days:", df['is_event'].sum())
df[df['is_event'] == 1][['date', 'event_name_1', 'event_type_1', 'is_sporting_event']].drop_duplicates().head(10)

Event features created
Total event days: 31600


,date,event_name_1,event_type_1,is_sporting_event
8,2011-02-06,SuperBowl,Sporting,1
16,2011-02-14,ValentinesDay,Cultural,0
23,2011-02-21,PresidentsDay,National,0
39,2011-03-09,LentStart,Religious,0
46,2011-03-16,LentWeek2,Religious,0
47,2011-03-17,StPatricksDay,Cultural,0
50,2011-03-20,Purim End,Religious,0
85,2011-04-24,OrthodoxEaster,Religious,0
87,2011-04-26,Pesach End,Religious,0
96,2011-05-05,Cinco De Mayo,Cultural,0


In [6]:
# Fill missing prices forward within each item
df['sell_price'] = df.groupby('id')['sell_price'].transform(
    lambda x: x.fillna(method='ffill')
)

# Price change from last week — did price drop or rise?
df['price_change'] = df.groupby('id')['sell_price'].transform(
    lambda x: x.diff()
)

# Price relative to item's own average — is this item cheap or expensive right now?
df['price_vs_avg'] = df.groupby('id')['sell_price'].transform(
    lambda x: x / x.mean()
)

print("Price features created")
df[['id', 'date', 'sell_price', 'price_change', 'price_vs_avg']].head(10)

C:\Users\Aadya Kapoor\AppData\Local\Temp\ipykernel_23964\1196775178.py:3: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  lambda x: x.fillna(method='ffill')


Price features created


,id,date,sell_price,price_change,price_vs_avg
0,FOODS_1_218_CA_1_evaluation,2011-01-29,1.0,NaN,1.018983
1,FOODS_1_218_CA_1_evaluation,2011-01-30,1.0,0.0,1.018983
2,FOODS_1_218_CA_1_evaluation,2011-01-31,1.0,0.0,1.018983
3,FOODS_1_218_CA_1_evaluation,2011-02-01,1.0,0.0,1.018983
4,FOODS_1_218_CA_1_evaluation,2011-02-02,1.0,0.0,1.018983
5,FOODS_1_218_CA_1_evaluation,2011-02-03,1.0,0.0,1.018983
6,FOODS_1_218_CA_1_evaluation,2011-02-04,1.0,0.0,1.018983
7,FOODS_1_218_CA_1_evaluation,2011-02-05,1.0,0.0,1.018983
8,FOODS_1_218_CA_1_evaluation,2011-02-06,1.0,0.0,1.018983
9,FOODS_1_218_CA_1_evaluation,2011-02-07,1.0,0.0,1.018983


In [7]:
# Already have snap_CA — just make sure it's int
df['snap_CA'] = df['snap_CA'].astype(int)

# Days until next SNAP period (first 3 days of month are SNAP days in CA)
# Simple proxy — is it early month?
df['is_early_month'] = (df['day_of_month'] <= 3).astype(int)

print("SNAP feature confirmed")
print("SNAP days in dataset:", df['snap_CA'].sum())

SNAP feature confirmed
SNAP days in dataset: 128000


In [8]:
# First 28 days per item will have NaN lags — drop them
before = len(df)
df.dropna(subset=['lag_7', 'lag_14', 'lag_28',
                   'rolling_mean_7', 'rolling_mean_28'], inplace=True)
after = len(df)

print(f"Rows before: {before}")
print(f"Rows after: {after}")
print(f"Dropped: {before - after} rows (expected — first 28 days per item)")

Rows before: 388200
Rows after: 382600
Dropped: 5600 rows (expected — first 28 days per item)
